# Libraries

In [59]:
import json
import time
import numpy as np
from pathlib import Path
from typing import List, Dict, Any
import re
from collections import defaultdict
from sentence_transformers import SentenceTransformer

try:
    import faiss
    print("✓ FAISS loaded")
except ImportError:
    raise ImportError("Run: pip install faiss-cpu")

print("✓ All libraries loaded")

✓ FAISS loaded
✓ All libraries loaded


# Paths

In [2]:
BASE_DIR    = Path("../")
CHUNK_DIR   = BASE_DIR / "data" / "chunks"
EMB_DIR     = BASE_DIR / "data" / "gold"
INDEX_DIR   = BASE_DIR / "data" / "indexes"
META_DIR    = BASE_DIR / "data" / "metadata"

EMB_DIR.mkdir(parents=True, exist_ok=True)
INDEX_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_FILES = {
    "small_fixed": CHUNK_DIR / "strategy_A_small_fixed.json",
    "large_fixed": CHUNK_DIR / "strategy_B_large_fixed.json",
    "semantic":    CHUNK_DIR / "strategy_C_semantic.json",
}

MODELS = {
    "minilm": "sentence-transformers/all-MiniLM-L6-v2",
    "bge":    "BAAI/bge-small-en-v1.5",
}

print("✓ Paths configured")
for name, path in CHUNK_FILES.items():
    exists = "✓" if path.exists() else "✗ MISSING"
    print(f"  {exists} {name}: {path.name}")

✓ Paths configured
  ✓ small_fixed: strategy_A_small_fixed.json
  ✓ large_fixed: strategy_B_large_fixed.json
  ✓ semantic: strategy_C_semantic.json


# Load Chunks

In [3]:
chunks_by_strategy: Dict[str, List[Dict]] = {}

for strategy_name, path in CHUNK_FILES.items():
    with open(path, encoding="utf-8") as f:
        chunks = json.load(f)
    chunks_by_strategy[strategy_name] = chunks
    print(f"  ✓ {strategy_name}: {len(chunks)} chunks loaded")

print(f"\nTotal chunks across all strategies: "
      f"{sum(len(v) for v in chunks_by_strategy.values())}")

  ✓ small_fixed: 1515 chunks loaded
  ✓ large_fixed: 595 chunks loaded
  ✓ semantic: 791 chunks loaded

Total chunks across all strategies: 2901


# Load Embedding Models

In [4]:
models: Dict[str, SentenceTransformer] = {}

for model_key, model_name in MODELS.items():
    print(f"  Loading {model_name} ...")
    t0 = time.time()
    models[model_key] = SentenceTransformer(model_name)
    print(f"  ✓ {model_key} loaded in {time.time() - t0:.1f}s")

print("\n✓ Both models ready")

  Loading sentence-transformers/all-MiniLM-L6-v2 ...


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7128.46it/s]


  ✓ minilm loaded in 14.9s
  Loading BAAI/bge-small-en-v1.5 ...


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5200.80it/s]


  ✓ bge loaded in 18.9s

✓ Both models ready


# Generate Embeddings — 6 Combinations

In [5]:
def generate_and_save_embeddings(
    model: SentenceTransformer,
    chunks: List[Dict],
    save_path: Path,
    batch_size: int = 64,
) -> np.ndarray:
    """
    Encode chunk texts, L2-normalize, and save to disk.
    Returns the normalized embedding matrix (n_chunks, dim).
    """
    texts = [c["text"] for c in chunks]

    t0 = time.time()
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,   # L2-normalize here
    )
    elapsed = time.time() - t0

    np.save(save_path, embeddings)
    print(f"    Shape: {embeddings.shape} | Time: {elapsed:.1f}s | "
          f"Speed: {len(texts)/elapsed:.0f} chunks/s → saved to {save_path.name}")

    return embeddings


# --- Generate all 6 combinations ---
embeddings_store: Dict[str, np.ndarray] = {}  # key: "minilm_small_fixed", etc.

for model_key, model in models.items():
    print(f"\n{'='*55}")
    print(f"Model: {model_key}")
    print(f"{'='*55}")
    for strategy_name, chunks in chunks_by_strategy.items():
        key       = f"{model_key}_{strategy_name}"
        save_path = EMB_DIR / f"{key}.npy"

        if save_path.exists():
            # Load cached embeddings to avoid recomputing
            embeddings = np.load(save_path)
            print(f"  ✓ {key}: loaded from cache {embeddings.shape}")
        else:
            print(f"  Encoding {key} ({len(chunks)} chunks)...")
            embeddings = generate_and_save_embeddings(model, chunks, save_path)

        embeddings_store[key] = embeddings

print(f"\n✓ {len(embeddings_store)} embedding matrices ready")


Model: minilm
  Encoding minilm_small_fixed (1515 chunks)...


Batches: 100%|██████████| 24/24 [00:20<00:00,  1.17it/s]


    Shape: (1515, 384) | Time: 20.6s | Speed: 74 chunks/s → saved to minilm_small_fixed.npy
  Encoding minilm_large_fixed (595 chunks)...


Batches: 100%|██████████| 10/10 [00:18<00:00,  1.89s/it]


    Shape: (595, 384) | Time: 18.9s | Speed: 31 chunks/s → saved to minilm_large_fixed.npy
  Encoding minilm_semantic (791 chunks)...


...

    Shape: (1515, 384) | Time: 38.6s | Speed: 39 chunks/s → saved to bge_small_fixed.npy
  Encoding bge_large_fixed (595 chunks)...


Batches: 100%|██████████| 10/10 [00:38<00:00,  3.87s/it]


    Shape: (595, 384) | Time: 38.7s | Speed: 15 chunks/s → saved to bge_large_fixed.npy
  Encoding bge_semantic (791 chunks)...


Batches: 100%|██████████| 13/13 [00:41<00:00,  3.18s/it]

    Shape: (791, 384) | Time: 41.4s | Speed: 19 chunks/s → saved to bge_semantic.npy

✓ 6 embedding matrices ready


# Build FAISS Indexes

In [6]:
faiss_indexes: Dict[str, faiss.IndexFlatIP] = {}

for key, embeddings in embeddings_store.items():
    dim        = embeddings.shape[1]
    index      = faiss.IndexFlatIP(dim)
    index.add(embeddings.astype(np.float32))
    faiss_indexes[key] = index

    save_path = INDEX_DIR / f"{key}.faiss"
    faiss.write_index(index, str(save_path))

    print(f"  ✓ {key}: {index.ntotal} vectors, dim={dim} → {save_path.name}")

print(f"\n✓ {len(faiss_indexes)} FAISS indexes built and saved")

  ✓ minilm_small_fixed: 1515 vectors, dim=384 → minilm_small_fixed.faiss
  ✓ minilm_large_fixed: 595 vectors, dim=384 → minilm_large_fixed.faiss
  ✓ minilm_semantic: 791 vectors, dim=384 → minilm_semantic.faiss
  ✓ bge_small_fixed: 1515 vectors, dim=384 → bge_small_fixed.faiss
  ✓ bge_large_fixed: 595 vectors, dim=384 → bge_large_fixed.faiss
  ✓ bge_semantic: 791 vectors, dim=384 → bge_semantic.faiss

✓ 6 FAISS indexes built and saved


# Retrieval Pipeline

In [ ]:
def retrieve(
    query: str,
    model_key: str,
    strategy_name: str,
    k: int = 5,
) -> List[Dict[str, Any]]:
    """
    Retrieve the top-K chunks most similar to the query.

    Returns a list of dicts with the chunk data plus a 'score' field.
    """
    index_key = f"{model_key}_{strategy_name}"
    model     = models[model_key]
    index     = faiss_indexes[index_key]
    chunks    = chunks_by_strategy[strategy_name]

    # Encode query (normalize for cosine similarity)
    q_vec = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)

    # FAISS search
    scores, indices = index.search(q_vec, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:   # FAISS returns -1 for empty slots
            continue
        result = dict(chunks[idx])  # copy to avoid mutation
        result["score"] = float(score)
        results.append(result)

    return results


#  Quick smoke test 
test_query   = "what is the mathematical formulation of residual functions with reference to layer inputs?"
test_results = retrieve(test_query, model_key="minilm", strategy_name="semantic", k=3)

print(f"Query: '{test_query}'\n")
for i, r in enumerate(test_results, 1):
    print(f"  [{i}] score={r['score']:.4f} | Paper: {r['paper_id']} | ID: {r['chunk_id']}")
    print(f"      Text snippet: {r['text'][:]}...")

Query: 'what is the mathematical formulation of residual functions with reference to layer inputs?'

  [1] score=0.6455 | Paper: 1512.03385v1 | ID: 1512.03385v1_0013
      Text snippet: Residual Learning
Let us consider H(x) as an underlying mapping to be
fit by a few stacked layers (not necessarily the entire net),
with x denoting the inputs to the first of these layers. If one
hypothesizes that multiple nonlinear layers can asymptotically approximate complicated functions2, then it is equivalent to hypothesize that they can asymptotically approximate the residual functions, i.e., H(x) −x (assuming that
the input and output are of the same dimensions). So
rather than expect stacked layers to approximate H(x), we
explicitly let these layers approximate a residual function
F(x) := H(x) −x. The original function thus becomes
F(x)+x. Although both forms should be able to asymptotically approximate the desired functions (as hypothesized),
the ease of learning might be different. This refor

# Latency Measurement

In [62]:
BENCHMARK_PATH = Path("../data/benchmark/benchmark.json")
BENCHMARK = json.loads(BENCHMARK_PATH.read_text(encoding="utf-8"))

def run_latency_test(benchmark, models, faiss_indexes, repeats=5):
    latency_data = []

    for model_key, model in models.items():
        for strategy_name in chunks_by_strategy:
            index_key = f"{model_key}_{strategy_name}"
            index = faiss_indexes[index_key]
            
            # Almacenar latencias por dificultad
            metrics = defaultdict(lambda: {"emb": [], "search": []})
            
            # Ejecutar benchmark repetidas veces
            for _ in range(repeats):
                for item in benchmark:
                    diff = item["difficulty"]
                    
                    # Medir Embedding
                    t0 = time.perf_counter()
                    q_vec = model.encode([item["query"]], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
                    emb_t = time.perf_counter() - t0
                    
                    # Medir Search
                    t0 = time.perf_counter()
                    index.search(q_vec, 5)
                    search_t = time.perf_counter() - t0
                    
                    metrics[diff]["emb"].append(emb_t)
                    metrics[diff]["search"].append(search_t)
            
            # Consolidar resultados
            for diff, times in metrics.items():
                latency_data.append({
                    "config": index_key,
                    "difficulty": diff,
                    "emb_ms": round(np.mean(times["emb"]) * 1000, 2),
                    "search_ms": round(np.mean(times["search"]) * 1000, 3)
                })
    return latency_data

# Ejecución
results = run_latency_test(BENCHMARK, models, faiss_indexes)
def run_latency_test(benchmark: List[Dict], models: Dict, faiss_indexes: Dict, repeats: int = 5):
    """
    Measures latency for embedding generation and vector search, 
    broken down by query difficulty.
    """
    latency_data = []

    for model_key, model in models.items():
        for strategy_name in chunks_by_strategy:
            index_key = f"{model_key}_{strategy_name}"
            index = faiss_indexes[index_key]
            
            # Temporary storage to aggregate times across repeats
            metrics = defaultdict(lambda: {"emb": [], "search": []})
            
            for _ in range(repeats):
                for item in benchmark:
                    diff = item["difficulty"]
                    
                    # Measure Embedding (Inference)
                    t0 = time.perf_counter()
                    q_vec = model.encode(
                        [item["query"]], 
                        convert_to_numpy=True, 
                        normalize_embeddings=True
                    ).astype(np.float32)
                    emb_t = time.perf_counter() - t0
                    
                    # Measure Vector Search (FAISS)
                    t0 = time.perf_counter()
                    index.search(q_vec, 5)
                    search_t = time.perf_counter() - t0
                    
                    metrics[diff]["emb"].append(emb_t)
                    metrics[diff]["search"].append(search_t)
            
            # Calculate means per difficulty
            for diff, times in metrics.items():
                latency_data.append({
                    "config": index_key,
                    "difficulty": diff,
                    "emb_ms": round(np.mean(times["emb"]) * 1000, 2),
                    "search_ms": round(np.mean(times["search"]) * 1000, 3)
                })
    return latency_data

# Run the test
latency_results = run_latency_test(BENCHMARK, models, faiss_indexes)

print(f"{'Config':<30} | {'Diff':<8} | {'Embed (ms)':>10} | {'Search (ms)':>12}")
print("-" * 75)

# Sort by config then difficulty for a cleaner view
for r in sorted(latency_results, key=lambda x: (x['config'], x['difficulty'])):
    print(f"{r['config']:<30} | {r['difficulty']:<8} | {r['emb_ms']:>10.2f} | {r['search_ms']:>12.3f}")

Config                         | Diff     | Embed (ms) |  Search (ms)
---------------------------------------------------------------------------
bge_large_fixed                | easy     |      28.15 |        0.125
bge_large_fixed                | hard     |      27.98 |        0.130
bge_large_fixed                | medium   |      27.82 |        0.135
bge_semantic                   | easy     |      27.99 |        0.156
bge_semantic                   | hard     |      28.26 |        0.160
bge_semantic                   | medium   |      29.03 |        0.160
bge_small_fixed                | easy     |      28.25 |        0.257
bge_small_fixed                | hard     |      28.33 |        0.255
bge_small_fixed                | medium   |      29.72 |        0.261
minilm_large_fixed             | easy     |      15.74 |        0.141
minilm_large_fixed             | hard     |      15.63 |        0.147
minilm_large_fixed             | medium   |      15.74 |        0.145
minilm_semanti

# Evaluation — Recall@K, Precision@K, MRR and Evaluation Proposal

In [ ]:
# Validate benchmark schema
REQUIRED_KEYS = {"query", "relevant_paper_id", "answer_contains", "difficulty"}
for i, item in enumerate(BENCHMARK):
    missing = REQUIRED_KEYS - set(item.keys())
    assert not missing, f"Item {i} is missing fields: {missing}"

print(f"✓ Benchmark loaded: {len(BENCHMARK)} queries")

# Adaptive Relevance Scoring (ARS)
def get_ars_score(chunk: dict, item: dict) -> float:
    """Calculates an adaptive score based on lexical, technical, and semantic signals."""
    if chunk["paper_id"] != item["relevant_paper_id"]:
        return 0.0
    
    chunk_text = chunk["text"].lower()
    
    # Lexical: Keyword overlap (ignoring stop words)
    stop_words = {"we", "the", "a", "is", "as", "with", "of", "to", "in", "by", "for", "that"}
    keywords = [w for w in re.findall(r'\w+', item["answer_contains"].lower()) if w not in stop_words]
    matches = sum(1 for kw in keywords if kw in chunk_text)
    lexical_score = (matches / len(keywords)) if keywords else 0.0
    
    # Technical: Heuristic for math/technical notation
    tech_patterns = ["y =", "f(x)", "equation", "formula", "wi", "x", "residual", "loss"]
    tech_matches = sum(1 for p in tech_patterns if p in chunk_text)
    technical_score = min(tech_matches / 4, 1.0)
    
    # Semantic: Vector similarity
    semantic_score = chunk.get("score", 0.5)
    
    # Weighted final score
    return (0.35 * lexical_score) + (0.25 * technical_score) + (0.4 * semantic_score)

def is_relevant(chunk: dict, item: dict, threshold: float = 0.45) -> bool:
    return get_ars_score(chunk, item) >= threshold

# Metrics Calculation 
def compute_metrics(
    query: str, benchmark_item: dict, model_key: str, strategy: str, k_values: List[int] = [3, 5]
) -> Dict[str, float]:
    
    results = retrieve(query, model_key, strategy, k=max(k_values))
    metrics = {}

    # MRR: 1 / rank of first relevant result
    metrics["mrr"] = 0.0
    for rank, chunk in enumerate(results, start=1):
        if is_relevant(chunk, benchmark_item):
            metrics["mrr"] = 1.0 / rank
            break

    # Recall & Precision
    for k in k_values:
        top_k = results[:k]
        hits = sum(1 for chunk in top_k if is_relevant(chunk, benchmark_item))
        metrics[f"recall@{k}"] = 1 if hits > 0 else 0
        metrics[f"precision@{k}"] = hits / k

    return metrics

✓ Benchmark loaded: 30 queries


In [54]:
eval_results = []
detail_results = []

for model in models:
    for strategy in chunks_by_strategy:
        config_name = f"{model}_{strategy}"
        config_metrics = []
        diff_bins = {"easy": [], "medium": [], "hard": []}

        for item in BENCHMARK:
            m = compute_metrics(item["query"], item, model, strategy)
            m.update({
                "query": item["query"], 
                "difficulty": item["difficulty"], 
                "config": config_name
            })
            
            config_metrics.append(m)
            diff_bins[item["difficulty"]].append(m)
            detail_results.append(m)

        # Aggregate Results
        summary = {"config": config_name, "model": model, "strategy": strategy}
        for m_key in ["mrr", "recall@3", "recall@5", "precision@5"]:
            summary[m_key] = round(np.mean([q[m_key] for q in config_metrics]), 4)
        
        # Breakdown by difficulty
        for diff in ["easy", "medium", "hard"]:
            items = diff_bins[diff]
            summary[f"mrr_{diff}"] = round(np.mean([q["mrr"] for q in items]), 4) if items else 0

        eval_results.append(summary)

# Formatted table including Easy, Medium, and Hard MRR
print(f"{'Config':<30} | {'R@3':>6} | {'R@5':>6} | {'P@5':>6} | {'MRR':>6} | {'Easy':>6} | {'Med':>6} | {'Hard':>6}")
print("-" * 105)

for r in eval_results:
    print(f"{r['config']:<30} | {r['recall@3']:>6.3f} | {r['recall@5']:>6.3f} | "
          f"{r['precision@5']:>6.3f} | {r['mrr']:>6.3f} | {r['mrr_easy']:>6.3f} | "
          f"{r['mrr_medium']:>6.3f} | {r['mrr_hard']:>6.3f}")

# Save results to disk
with open(META_DIR / "retrieval_eval.json", "w") as f:
    json.dump(eval_results, f, indent=2)

Config                         |    R@3 |    R@5 |    P@5 |    MRR |   Easy |    Med |   Hard
---------------------------------------------------------------------------------------------------------
minilm_small_fixed             |  0.667 |  0.700 |  0.340 |  0.596 |  0.620 |  0.722 |  0.375
minilm_large_fixed             |  0.833 |  0.867 |  0.553 |  0.751 |  0.900 |  0.917 |  0.317
minilm_semantic                |  0.733 |  0.800 |  0.447 |  0.617 |  0.750 |  0.757 |  0.240
bge_small_fixed                |  0.933 |  0.933 |  0.693 |  0.917 |  0.950 |  1.000 |  0.750
bge_large_fixed                |  0.867 |  0.867 |  0.700 |  0.800 |  0.900 |  0.875 |  0.562
bge_semantic                   |  0.867 |  0.900 |  0.693 |  0.812 |  1.000 |  0.808 |  0.583
